# Notebook 2: CFD Pipeline — Automated SAS Segmentation & Mesh Export

**Goal:** Demonstrate the end-to-end CFD segmentation pipeline:
1. Load spine-generic T2w volume
2. Run pre-trained models (TotalSpineSeg, model-canal-seg, RootletSeg)
3. Boolean CSF domain extraction: `canal − cord − rootlets`
4. Mesh export with watertightness validation
5. Geometric validation against Sass 2017 reference
6. CSF flow waveform visualization (boundary conditions)

## Pipeline Architecture
```
T2w MRI → TotalSpineSeg (cord + canal)
        → model-canal-seg (dural sac)
        → RootletSeg (nerve rootlets)
        → Boolean: CSF = canal − cord − rootlets
        → Mesh export (marching cubes + repair + smooth)
        → Validate vs Sass 2017 (97.3 cm³, Re=68.5, Wo=9.6)
```

In [ ]:
import sys
from pathlib import Path

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Project root
PROJECT_ROOT = Path("..")
sys.path.insert(0, str(PROJECT_ROOT))

# Import pipeline modules
from src.segmentation.model_registry import MODELS, list_models, get_model, check_tool_available, SegTool
from src.mesh.cfd_domain import extract_csf_domain, compute_cross_sectional_metrics, DomainMetrics
from src.mesh.export import MeshExportConfig, extract_surface, repair_mesh, validate_mesh, export_cfd_mesh
from src.evaluation.geometry_metrics import validate_against_sass, SASS_REF, compute_hydraulic_diameter_profile
from src.cfd.boundary_conditions import (
    get_sass_waveform_c2c3, get_sass_waveform_c7t1, get_sass_waveform_t10t11,
    interpolate_flow_at_level
)

# Data paths
DATA_ROOT = PROJECT_ROOT / "data" / "spine-generic"
LABELS_ROOT = DATA_ROOT / "derivatives" / "labels"
REF_GEOMETRY = PROJECT_ROOT / "data" / "reference_geometry" / "12987_2017_85_MOESM1_ESM"

print(f"Data root:        {DATA_ROOT.resolve()}")
print(f"Labels root:      {LABELS_ROOT.resolve()}")
print(f"Reference meshes: {REF_GEOMETRY.resolve()}")
print(f"Ref meshes exist: {REF_GEOMETRY.exists()}")

## 1. Model Registry — Pre-trained Models for CFD Segmentation

The pipeline uses existing battle-tested models rather than training from scratch:
- **TotalSpineSeg** — cord + canal + vertebrae (whole-spine, any contrast)
- **model-canal-seg** — dural sac outer boundary (T2w specific)
- **RootletSeg** — nerve rootlets C2-T1 (Dice 0.67±0.16)
- **SCT contrast-agnostic** — spinal cord only (any contrast)

In [ ]:
# Show all registered models
print("═" * 80)
print(f"{'Model':<25} {'Tool':<18} {'Structures':<40} {'Version'}")
print("═" * 80)

for name in list_models():
    m = get_model(name)
    structs = ", ".join(m.structures[:3])  # truncate for display
    print(f"{name:<25} {m.tool.value:<18} {structs:<40} {m.version}")

print("\n--- Tool Availability ---")
for tool in SegTool:
    available = check_tool_available(tool)
    status = "✓ installed" if available else "✗ not found"
    print(f"  {tool.value:<25} {status}")

## 2. Sass 2017 Reference Geometry

The benchmark for validating our automated pipeline is the Sass et al. 2017 model:
- Subject: healthy 23-year-old female
- Coverage: foramen magnum to S5 (complete spinal SAS)
- Structures: dura, cord, 31 rootlet pairs
- **Total CSF volume: 97.3 cm³**
- License: CC-BY-SA-4.0

In [ ]:
# Display Sass 2017 reference values
print("╔══════════════════════════════════════════════════════════════╗")
print("║  SASS 2017 REFERENCE — Spinal Subarachnoid Space Model     ║")
print("╚══════════════════════════════════════════════════════════════╝")
print(f"\n  Volumes:")
print(f"    CSF (total):      {SASS_REF.csf_volume_cm3} cm³")
print(f"    Dura:             {SASS_REF.dura_volume_cm3} cm³")
print(f"    Spinal cord:      {SASS_REF.cord_volume_cm3} cm³")
print(f"    Nerve rootlets:   {SASS_REF.rootlet_volume_cm3} cm³")
print(f"\n  Surface Areas:")
print(f"    Dura:             {SASS_REF.dura_surface_area_cm2} cm²")
print(f"    Cord:             {SASS_REF.cord_surface_area_cm2} cm²")
print(f"    Rootlets:         {SASS_REF.rootlet_surface_area_cm2} cm²")
print(f"\n  Hydrodynamic Parameters:")
print(f"    Max Reynolds:     {SASS_REF.max_reynolds} (at C3-C4)")
print(f"    Avg Reynolds:     {SASS_REF.avg_reynolds}")
print(f"    Avg Womersley:    {SASS_REF.avg_womersley}")
print(f"    Pulse wave vel:   {SASS_REF.pulse_wave_velocity} cm/s")
print(f"\n  Lengths:")
print(f"    Cord:             {SASS_REF.cord_length_cm} cm")
print(f"    Dura:             {SASS_REF.dura_length_cm} cm")

# Check reference STLs
if REF_GEOMETRY.exists():
    ref_files = list(REF_GEOMETRY.glob("*"))
    print(f"\n  Reference mesh files ({len(ref_files)}):")
    for f in sorted(ref_files):
        size_mb = f.stat().st_size / 1e6
        print(f"    {f.name:<25} ({size_mb:.1f} MB)")
else:
    print("\n  ⚠️  Reference geometry not found at expected path.")

## 3. CSF Domain Extraction Demo (on existing labels)

Since SCT may not be installed yet, we can demonstrate the Boolean extraction
using the spine-generic **canal** and **cord** labels that already exist in
the derivatives directory.

Formula: `CSF = canal − cord − rootlets`

In [ ]:
# Find a subject with both cord AND canal labels
def find_subject_with_canal_and_cord(labels_root, min_size=1000):
    """Find a subject that has both SC and canal segmentation labels."""
    for sub_dir in sorted(labels_root.iterdir()):
        if not sub_dir.is_dir() or not sub_dir.name.startswith("sub-"):
            continue
        sub_id = sub_dir.name
        anat_dir = sub_dir / "anat"
        if not anat_dir.exists():
            continue
        
        cord_path = anat_dir / f"{sub_id}_T2w_label-SC_seg.nii.gz"
        canal_path = anat_dir / f"{sub_id}_T2w_label-canal_seg.nii.gz"
        
        cord_ok = cord_path.exists() and cord_path.stat().st_size > min_size
        canal_ok = canal_path.exists() and canal_path.stat().st_size > min_size
        
        if cord_ok and canal_ok:
            return sub_id, cord_path, canal_path
    
    return None, None, None

sub_id, cord_path, canal_path = find_subject_with_canal_and_cord(LABELS_ROOT)

if sub_id:
    print(f"Found subject: {sub_id}")
    print(f"  Cord:  {cord_path}")
    print(f"  Canal: {canal_path}")
else:
    print("No subject found with both cord and canal labels.")
    print("This is expected if git-annex data hasn't been fetched.")
    print("\nFallback: We'll demonstrate with synthetic data below.")

In [ ]:
# Extract CSF domain from real labels (if available)
if cord_path and canal_path:
    csf_data, metrics = extract_csf_domain(
        canal_path=canal_path,
        cord_path=cord_path,
        rootlet_path=None,  # Rootlets not available in spine-generic
        keep_largest_component=True,
    )
    
    print("\n═══ CSF Domain Extraction Results ═══")
    print(f"  CSF volume:     {metrics.csf_volume_cm3:.2f} cm³")
    print(f"  Cord volume:    {metrics.cord_volume_cm3:.2f} cm³")
    print(f"  Canal volume:   {metrics.canal_volume_cm3:.2f} cm³")
    print(f"  Components:     {metrics.n_connected_components}")
    print(f"  Voxel spacing:  {metrics.voxel_spacing_mm} mm")
    print(f"\n  Sass ratio:     {metrics.csf_volume_cm3 / SASS_REF.csf_volume_cm3 * 100:.1f}% of reference")
    print(f"  (Note: spine-generic is cervical only, Sass covers full spine)")
else:
    # Create synthetic demo
    print("Creating synthetic demo data for visualization...")
    shape = (64, 64, 100)  # Simplified spine-like volume
    canal_data = np.zeros(shape, dtype=bool)
    cord_data = np.zeros(shape, dtype=bool)
    
    # Synthetic canal (elliptical tube)
    for z in range(10, 90):
        y, x = np.ogrid[-32:32, -32:32]
        canal_radius = 10 - 0.02 * abs(z - 50)  # Slight taper
        cord_radius = 4 - 0.01 * abs(z - 50)
        canal_data[:, :, z] = (x**2 + y**2) < canal_radius**2
        cord_data[:, :, z] = (x**2 + y**2) < cord_radius**2
    
    csf_data = np.logical_and(canal_data, ~cord_data).astype(np.uint8)
    metrics = DomainMetrics(
        csf_volume_cm3=csf_data.sum() * 0.001,  # 1mm³ voxels
        cord_volume_cm3=cord_data.sum() * 0.001,
        canal_volume_cm3=canal_data.sum() * 0.001,
    )
    print(f"  Synthetic CSF volume: {metrics.csf_volume_cm3:.1f} cm³")

In [ ]:
# Visualize the CSF domain extraction
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Pick representative slices
n_slices = csf_data.shape[2]
slice_indices = np.linspace(n_slices * 0.2, n_slices * 0.8, 4, dtype=int)

# Color map for multi-label
cmap_labels = ListedColormap(['black', 'red', 'cyan', 'yellow', 'green'])

for col, z in enumerate(slice_indices):
    # Top row: individual structures
    if cord_path:  # Real data
        canal_slice = np.asarray(nib.load(canal_path).dataobj)[:, :, z].T.astype(bool)
        cord_slice = np.asarray(nib.load(cord_path).dataobj)[:, :, z].T.astype(bool)
    else:  # Synthetic
        canal_slice = canal_data[:, :, z].T
        cord_slice = cord_data[:, :, z].T
    
    csf_slice = csf_data[:, :, z].T
    
    # Combined view (top row)
    combined = np.zeros_like(csf_slice, dtype=np.uint8)
    combined[canal_slice > 0] = 1  # Canal boundary
    combined[csf_slice > 0] = 2    # CSF domain
    combined[cord_slice > 0] = 3   # Cord
    
    axes[0, col].imshow(combined, cmap=cmap_labels, vmin=0, vmax=4,
                        origin='lower', aspect='equal')
    axes[0, col].set_title(f'Slice {z}', fontsize=10)
    axes[0, col].axis('off')
    
    # CSF only (bottom row)
    axes[1, col].imshow(csf_slice, cmap='Blues', origin='lower', aspect='equal')
    axes[1, col].set_title(f'CSF domain', fontsize=10)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Combined\n(canal/CSF/cord)', fontsize=10)
axes[1, 0].set_ylabel('CSF domain\n(fluid region)', fontsize=10)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label='Canal boundary'),
    Patch(facecolor='cyan', label='CSF domain (fluid)'),
    Patch(facecolor='yellow', label='Spinal cord'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11)

plt.suptitle(f'CSF Domain Extraction: canal − cord = fluid domain\n'
             f'Volume: {metrics.csf_volume_cm3:.1f} cm³ (Sass ref: {SASS_REF.csf_volume_cm3} cm³)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(bottom=0.08)
plt.show()

## 4. Cross-Sectional Metrics Along the Spine

These are validated against Sass 2017 Fig 7:
- Cross-sectional area (CSA) decreases caudally
- Hydraulic diameter: D_H = 4A/P
- Used to compute Reynolds and Womersley numbers

In [ ]:
# Compute cross-sectional metrics
spacing = metrics.voxel_spacing_mm if hasattr(metrics, 'voxel_spacing_mm') else (1.0, 1.0, 1.0)
cs_metrics = compute_cross_sectional_metrics(csf_data, spacing=spacing, axis=2)

# Plot profiles
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

positions = cs_metrics['slice_positions_mm']
csa = cs_metrics['cross_sectional_area_mm2']
dh = cs_metrics['hydraulic_diameter_mm']
perim = cs_metrics['perimeter_mm']

# Filter to non-zero region
mask = csa > 0

axes[0].plot(positions[mask], csa[mask], 'b-', linewidth=2)
axes[0].set_xlabel('Distance from top (mm)')
axes[0].set_ylabel('CSA (mm²)')
axes[0].set_title('Cross-Sectional Area')
axes[0].grid(True, alpha=0.3)

axes[1].plot(positions[mask], dh[mask], 'r-', linewidth=2)
axes[1].set_xlabel('Distance from top (mm)')
axes[1].set_ylabel('D_H (mm)')
axes[1].set_title('Hydraulic Diameter (4A/P)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(positions[mask], perim[mask], 'g-', linewidth=2)
axes[2].set_xlabel('Distance from top (mm)')
axes[2].set_ylabel('Perimeter (mm)')
axes[2].set_title('Wetted Perimeter')
axes[2].grid(True, alpha=0.3)

plt.suptitle('CSF Domain — Axial Profile (compare to Sass 2017 Fig 7)', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"CSA: mean={csa[mask].mean():.1f} mm², max={csa[mask].max():.1f} mm²")
print(f"D_H: mean={dh[mask].mean():.2f} mm")
print(f"Sass ref avg CSA of SSS ≈ 160 mm² (1.60 cm²)")

## 5. CSF Flow Boundary Conditions (Sass 2017 Waveforms)

Before internal PC-MRI data is acquired, we use published waveforms from
Sass 2017 (Fig 6a) as OpenFOAM boundary conditions:
- **C2-C3:** peak 4.75 cm³/s (4.0 cm from foramen magnum)
- **C7-T1:** peak 3.05 cm³/s (12.5 cm from FM)
- **T10-T11:** peak 1.26 cm³/s (35.4 cm from FM)

Flow is oscillatory: caudal during systole, cranial during diastole.
Zero net flow over one cardiac cycle (~800 ms at 75 bpm).

In [ ]:
# Get Sass waveforms at three measurement levels
wf_c2 = get_sass_waveform_c2c3(n_points=200)
wf_c7 = get_sass_waveform_c7t1(n_points=200)
wf_t10 = get_sass_waveform_t10t11(n_points=200)

# Also interpolate at mid-thoracic
wf_mid = interpolate_flow_at_level(25.0, n_points=200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Individual waveforms
ax = axes[0]
ax.plot(wf_c2.time_s * 1000, wf_c2.flow_cm3_per_s, 'b-', linewidth=2, label=f'C2-C3 ({wf_c2.distance_from_fm_cm} cm)')
ax.plot(wf_c7.time_s * 1000, wf_c7.flow_cm3_per_s, 'r-', linewidth=2, label=f'C7-T1 ({wf_c7.distance_from_fm_cm} cm)')
ax.plot(wf_mid.time_s * 1000, wf_mid.flow_cm3_per_s, 'g--', linewidth=1.5, label=f'Mid-thoracic (25 cm)')
ax.plot(wf_t10.time_s * 1000, wf_t10.flow_cm3_per_s, 'm-', linewidth=2, label=f'T10-T11 ({wf_t10.distance_from_fm_cm} cm)')
ax.axhline(0, color='gray', linestyle='-', linewidth=0.5)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('CSF Flow Rate (cm³/s)')
ax.set_title('CSF Flow Waveforms (Sass 2017)\n(negative = caudal/systolic)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 800])

# Right: Peak flow amplitude vs distance from FM
ax = axes[1]
distances = [wf_c2.distance_from_fm_cm, wf_c7.distance_from_fm_cm, 
             wf_mid.distance_from_fm_cm, wf_t10.distance_from_fm_cm]
peaks = [wf_c2.peak_caudal_flow, wf_c7.peak_caudal_flow,
         wf_mid.peak_caudal_flow, wf_t10.peak_caudal_flow]

# Interpolated curve
d_range = np.linspace(2, 55, 50)
peak_curve = [interpolate_flow_at_level(d).peak_caudal_flow for d in d_range]

ax.plot(d_range, peak_curve, 'k-', linewidth=1.5, label='Interpolated')
ax.scatter(distances, peaks, s=100, c=['blue', 'red', 'green', 'magenta'], zorder=5)
ax.set_xlabel('Distance from Foramen Magnum (cm)')
ax.set_ylabel('Peak Caudal Flow (cm³/s)')
ax.set_title('CSF Flow Amplitude Decay Along Spine')
ax.grid(True, alpha=0.3)
ax.legend()

# Annotate
for d, p, label in zip(distances, peaks, ['C2-C3', 'C7-T1', 'Mid-T', 'T10-T11']):
    ax.annotate(label, (d, p), textcoords='offset points', xytext=(5, 10), fontsize=9)

plt.tight_layout()
plt.show()

print("\nKey flow parameters for OpenFOAM BCs:")
print(f"  C2-C3:   peak caudal = {wf_c2.peak_caudal_flow:.2f} cm³/s")
print(f"  C7-T1:   peak caudal = {wf_c7.peak_caudal_flow:.2f} cm³/s")
print(f"  T10-T11: peak caudal = {wf_t10.peak_caudal_flow:.2f} cm³/s")
print(f"\nSass published peaks: 4.75, 3.05, 1.26 cm³/s")
print(f"(Our sinusoidal approximation captures the amplitude; exact shape\n"
      f" requires digitizing from Fig 6a or acquiring internal PC-MRI)")

## 6. Mesh Export & Quality Validation

CFD-ready meshes must be:
- **Watertight** — no holes (fluid leaks → solver crash)
- **Manifold** — consistent winding, every edge shared by exactly 2 faces
- **Euler χ = 2** — topological sphere (genus-0 surface)
- **No degenerate faces** — zero-area triangles prevent BL extrusion

In [ ]:
try:
    import trimesh
    TRIMESH_OK = True
except ImportError:
    TRIMESH_OK = False
    print("trimesh not installed. Install with: pip install trimesh")

if TRIMESH_OK and csf_data.sum() > 0:
    from src.mesh.export import extract_surface, repair_mesh, validate_mesh, MeshExportConfig
    
    # Save CSF domain to temp file for mesh extraction
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.nii.gz', delete=False) as f:
        temp_path = Path(f.name)
    
    # Create NIfTI from CSF data
    affine = np.eye(4)
    for i, s in enumerate(spacing):
        affine[i, i] = s
    nii = nib.Nifti1Image(csf_data.astype(np.uint8), affine)
    nib.save(nii, temp_path)
    
    # Extract and repair mesh
    config = MeshExportConfig(
        smooth_iterations=10,
        smooth_lambda=0.5,
        smooth_mu=-0.53,
    )
    
    print("Extracting surface mesh...")
    mesh = extract_surface(temp_path, config)
    
    if mesh is not None:
        print(f"  Raw mesh: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces")
        print(f"  Watertight: {mesh.is_watertight}")
        
        print("\nRepairing mesh...")
        mesh = repair_mesh(mesh, config)
        
        print("\nValidating...")
        quality = validate_mesh(mesh)
        
        print(f"\n{'═' * 50}")
        print(f"  MESH QUALITY REPORT")
        print(f"{'═' * 50}")
        print(f"  Watertight:     {quality.is_watertight}  {'✓' if quality.is_watertight else '✗'}")
        print(f"  Manifold:       {quality.is_manifold}  {'✓' if quality.is_manifold else '✗'}")
        print(f"  Euler number:   {quality.euler_number}  {'✓' if quality.euler_number == 2 else '✗ (expect 2)'}")
        print(f"  Vertices:       {quality.vertex_count}")
        print(f"  Faces:          {quality.face_count}")
        print(f"  Volume:         {quality.volume_cm3:.2f} cm³")
        print(f"  Degenerate:     {quality.has_degenerate_faces}")
        print(f"  CFD-READY:      {quality.passes_cfd_check}  {'✓' if quality.passes_cfd_check else '✗'}")
    
    # Cleanup
    temp_path.unlink(missing_ok=True)
else:
    print("Skipping mesh export (trimesh not available or empty CSF domain).")

## 7. Geometry Validation Against Sass Reference

In [ ]:
# Run validation
validation = validate_against_sass(
    csf_volume_cm3=metrics.csf_volume_cm3,
    cord_volume_cm3=metrics.cord_volume_cm3,
    csa_profile=cs_metrics['cross_sectional_area_mm2'] if 'cs_metrics' in dir() else None,
    volume_tolerance_percent=10.0,
)

print("╔══════════════════════════════════════════════════════════════╗")
print("║  GEOMETRY VALIDATION vs SASS 2017                          ║")
print("╚══════════════════════════════════════════════════════════════╝")
print(f"\n  CSF Volume:        {validation.csf_volume_cm3:.2f} cm³")
print(f"  Sass Reference:    {SASS_REF.csf_volume_cm3} cm³")
print(f"  Deviation:         {validation.volume_error_percent:.1f}%")
print(f"  Within tolerance:  {validation.volume_within_tolerance}")
print(f"\n  Overall pass:      {validation.passes_geometry_check}")
print(f"\n  Notes:")
for note in validation.notes:
    print(f"    {note}")

## 8. Full Pipeline Command (when SCT is installed)

Once Spinal Cord Toolbox is installed, run the complete pipeline:

```bash
# Install SCT (one-time)
# See: https://spinalcordtoolbox.com/user_section/installation.html

# Run on a single subject
python scripts/run_pipeline.py \
    --input data/spine-generic/sub-amu01/anat/sub-amu01_T2w.nii.gz \
    --output outputs/pipeline/ \
    --mesh

# Run batch on all subjects
python scripts/run_pipeline.py run-batch \
    --data data/spine-generic/ \
    --output outputs/batch/ \
    --max 10
```

This will produce:
- Per-subject segmentation masks (cord, canal, rootlets)
- CSF domain mask (Boolean extraction)
- Combined multi-label visualization
- Watertight STL mesh (CFD-ready)
- Quality metrics report

## Summary

**What we demonstrated:**
1. ✅ Model registry with version-locked pre-trained models
2. ✅ CSF domain extraction via Boolean subtraction
3. ✅ Cross-sectional geometric analysis (CSA, hydraulic diameter)
4. ✅ Sass 2017 reference values for validation
5. ✅ CSF flow waveforms for OpenFOAM boundary conditions
6. ✅ Mesh export with watertightness/manifoldness validation

**Next steps:**
- Install SCT to enable the full automated pipeline
- Download SPIDER dataset for lumbar coverage
- Run mesh independence study (3 refinement levels)
- Set up OpenFOAM case with exported mesh + Sass BCs

**Key numbers to remember:**
- Sass CSF volume: 97.3 cm³ (our target: ±10%)
- Max Re: 174.9 at C3-C4 (laminar flow confirmed)
- Avg Womersley: 9.6 (inertia-dominated pulsatile flow)
- CSF PWV: 19.4 cm/s
- Rootlets add +60% steady-streaming drug spread (Khani 2018)